# 레퍼런스 이미지 PoC (광고 이미지 만들기 2번)

사장님이 마음에 드는 광고 사진을 올리면 **그 분위기로** 광고를 만든다.

정공법 둘(IP-Adapter · img2img)을 먼저 재고, 둘 다 탈락한 뒤
"레퍼런스를 말로 바꿔 프롬프트에 얹는" 방식을 택했다.

GPU: Colab T4 / RTX 3060 Ti 8GB

## 1. IP-Adapter 는 sd-turbo 에 올라가지 않는다

IP-Adapter 는 모델의 cross-attention 차원에 맞춰 만들어진다.
sd-turbo(SD 2.1 계열)는 1024, 배포된 어댑터는 768(SD1.5)·2048(SDXL)이다.
**설정 문제가 아니라 구조 문제**라 같은 모델을 쓰는 한 방법이 없다.

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

probe = StableDiffusionPipeline.from_pretrained(
    "stabilityai/sd-turbo", torch_dtype=torch.float16, safety_checker=None
).to("cuda")

for sub, name in [
    ("models", "ip-adapter_sd15.bin"),
    ("models", "ip-adapter-plus_sd15.bin"),
    ("sdxl_models", "ip-adapter_sdxl.bin"),
]:
    try:
        probe.load_ip_adapter("h94/IP-Adapter", subfolder=sub, weight_name=name)
        print(f"올라감: {sub}/{name}")
    except Exception as e:
        print(f"실패 {sub}/{name}\n   {type(e).__name__}: {str(e)[:120]}\n")

```
sd15       expected [320, 1024]  got [320, 768]
sd15-plus  expected [320, 1024]  got [320, 768]
sdxl       expected [320, 1024]  got [640, 2048]
```

셋 다 실패. **IP-Adapter 는 후보에서 빠진다.**

## 2. img2img vs 말로 옮기기

분위기가 확실히 구분되는 레퍼런스를 만들고, **레퍼런스와 다른 상품**을 주문한다.
상품이 바뀌는지가 판별 기준이다 — 안 바뀌면 베끼는 것이다.

In [ ]:
from diffusers import AutoPipelineForImage2Image, AutoPipelineForText2Image
from PIL import Image

t2i = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sd-turbo", torch_dtype=torch.float16, safety_checker=None
).to("cuda")
i2i = AutoPipelineForImage2Image.from_pretrained(
    "stabilityai/sd-turbo", torch_dtype=torch.float16, safety_checker=None
).to("cuda")

REF_STYLE = "dark moody rustic wooden table, dim candlelight, deep shadows"
TARGET = "appetizing fried chicken on a plate"  # 레퍼런스는 커피 — 달라야 판별된다

ref = t2i(prompt=f"a cup of coffee, {REF_STYLE}", num_inference_steps=4,
          guidance_scale=0.0,
          generator=torch.Generator("cuda").manual_seed(7)).images[0]

outs = [("레퍼런스", ref)]
for s in (0.5, 0.7, 0.9):
    outs.append((f"A img2img {s}", i2i(
        prompt=TARGET, image=ref, strength=s,
        num_inference_steps=6, guidance_scale=0.0,   # strength×steps ≥ 3 이 되게
        generator=torch.Generator("cuda").manual_seed(0),
    ).images[0]))

outs.append(("B 말로", t2i(
    prompt=f"{TARGET}, {REF_STYLE}", num_inference_steps=4, guidance_scale=0.0,
    generator=torch.Generator("cuda").manual_seed(0),
).images[0]))

strip = Image.new("RGB", (512 * len(outs), 512))
for i, (_, img) in enumerate(outs):
    strip.paste(img, (512 * i, 0))
print(" | ".join(n for n, _ in outs))
strip

**결과**

| | 상품이 바뀌었나 | 분위기가 옮겨왔나 |
|---|---|---|
| img2img 0.5 | ❌ 커피 그대로 | — |
| img2img 0.7 | ❌ 커피 남고 치킨 조각만 | — |
| img2img 0.9 | ❌ 커피잔이 아직 있다 | — |
| **말로 옮김** | ✅ 치킨만 | ✅ 어두운 탁자·그림자 |

img2img 는 strength 를 0.9 까지 올려도 레퍼런스의 상품이 안 없어진다.
사장님 상품이 안 나오는 것도 문제지만, 그전에 **남의 광고를 복제**하는 것이 된다.

→ **말로 옮기는 방식으로 간다.** GPU 도 안 쓴다.

## 결론 — src/app_core/ref_style.py 로 옮긴 것

레퍼런스를 비전 모델로 읽어 **영어 분위기 구절**을 만들고 생성 프롬프트 뒤에 붙인다.

핵심 제약 두 가지가 프롬프트에 들어간다.

1. **상품·글자·사람을 말하지 마라.** 이게 무너지면 남의 상품이 사장님 광고로
   들어온다 — img2img 를 버린 이유가 정확히 그것이다.
2. **12단어 이내.** CLIP 이 77토큰에서 자른다. 분위기는 뒤에 붙여서, 잘릴 때
   상품·구도가 먼저 살아남게 한다.

### 남은 것

- 여기 레퍼런스는 모델이 만든 그림이다. **실제 사장님이 올릴 광고 사진**
  (글자·로고가 박힌)으로 다시 확인해야 한다 — 글자를 분위기로 읽어버릴 수 있다.
- 2번과 4번을 같이 쓰는 경우(레퍼런스 + 스케치)는 아직 안 재봤다.